<a href="https://colab.research.google.com/github/QueenWafa/grain-classification-cnn/blob/main/mit7074_redis_online_shopping_cart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# MIT7074 – ADVANCED DATABASE MANAGEMENT SYSTEM

## Practical Exercise: Redis Key-Value Store

### Project Title: Online Shopping Cart System

**Student Name:** Nourizadeh SeyedehFatemeh

**Student ID:** 20221010010

**Database Technology:** Redis

**NoSQL Model:** Key-Value

**Programming Language:** Python

**Development Environment:** Google Colab


In [ ]:
import sys

print("PROJECT: Redis Online Shopping Cart")
print("Python Version:", sys.version)
print("Development Environment: Google Colab")

print("\nEnvironment check completed.")


PROJECT: Redis Online Shopping Cart
Python Version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Development Environment: Google Colab

Environment check completed.


In [ ]:
!apt-get update -qq
!apt-get install -y redis-server


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libjemalloc2 liblzf1 redis-tools
Suggested packages:
  ruby-redis
The following NEW packages will be installed:
  libjemalloc2 liblzf1 redis-server redis-tools
0 upgraded, 4 newly installed, 0 to remove and 65 not upgraded.
Need to get 1,482 kB of archives.
After this operation, 7,568 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libjemalloc2 amd64 5.3.0-2build1 [256 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 liblzf1 amd64 3.6-4 [7,624 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 redis-tools amd64 5:7.0.15-1ubuntu0.24.04.5 [1,167 kB]
Get:4 http://arc

In [ ]:
!redis-server --daemonize yes
!redis-cli ping


PONG


In [ ]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 8.3 MB/s  0:00:00


In [ ]:

import redis

# Establish connection to Redis
r = redis.Redis(
    host="localhost",
    port=6379,
    db=0,
    decode_responses=True
)

# Test Redis connection
try:
    if r.ping():
        print("Successfully connected to Redis!")
        print("Database: Redis")
        print("Host: localhost")
        print("Port: 6379")
        print("Database Number: 0")

except redis.ConnectionError:
    print("Connection failed. Please restart Redis.")


Successfully connected to Redis!
Database: Redis
Host: localhost
Port: 6379
Database Number: 0



##  Redis Database Design and Key Structure
The Redis database uses a key-value model to organise shopping cart information. Each product is stored using a unique key with a Hash data structure. Product details such as product name, price, stock quantity and category are stored as fields within the Hash.


## Product Database Creation and Data Insertion


In [ ]:

# Product information for the online store

products = {
    "P001": {
        "name": "Wireless Mouse",
        "price": "29.90",
        "stock": "150",
        "category": "Electronics"
    },

    "P002": {
        "name": "Mechanical Keyboard",
        "price": "89.50",
        "stock": "80",
        "category": "Electronics"
    },

    "P003": {
        "name": "USB-C Cable",
        "price": "12.00",
        "stock": "500",
        "category": "Accessories"
    },

    "P004": {
        "name": "Laptop Stand",
        "price": "45.00",
        "stock": "60",
        "category": "Accessories"
    }
}

# Insert products into Redis
for product_id, details in products.items():

    r.hset(
        f"product:{product_id}",
        mapping=details
    )

    print(f"Product inserted successfully: {details['name']}")

print("\nAll products have been saved in Redis!")


Product inserted successfully: Wireless Mouse
Product inserted successfully: Mechanical Keyboard
Product inserted successfully: USB-C Cable
Product inserted successfully: Laptop Stand

All products have been saved in Redis!


name      = Wireless Mouse
price     = 29.90
stock     = 150
category  = Electronics


##  Retrieving Product Information from Redis


In [ ]:

# Retrieve all product records from Redis

print("========== PRODUCT DATABASE ==========\n")

for product_id in products:

    product_data = r.hgetall(f"product:{product_id}")

    print("Product ID:", product_id)
    print("Product Name:", product_data["name"])
    print("Price: RM", product_data["price"])
    print("Available Stock:", product_data["stock"])
    print("Category:", product_data["category"])
    print("--------------------------------------")


========== PRODUCT DATABASE ==========

Product ID: P001
Product Name: Wireless Mouse
Price: RM 29.90
Available Stock: 150
Category: Electronics
--------------------------------------
Product ID: P002
Product Name: Mechanical Keyboard
Price: RM 89.50
Available Stock: 80
Category: Electronics
--------------------------------------
Product ID: P003
Product Name: USB-C Cable
Price: RM 12.00
Available Stock: 500
Category: Accessories
--------------------------------------
Product ID: P004
Product Name: Laptop Stand
Price: RM 45.00
Available Stock: 60
Category: Accessories
--------------------------------------



##  Redis Database Verification


In [ ]:

# Display all product keys

product_keys = sorted(r.keys("product:*"))

print("PRODUCT KEYS IN REDIS")
print("=====================")

for key in product_keys:
    print(key)

print("\nTotal Products:", len(product_keys))

# Check Redis database information
print("\nTotal Keys in Database:", r.dbsize())


PRODUCT KEYS IN REDIS
product:P001
product:P002
product:P003
product:P004

Total Products: 4

Total Keys in Database: 4



##  Redis Server Information and Configuration


In [ ]:
# Retrieve Redis server information

server_info = r.info("server")
memory_info = r.info("memory")

print("========== REDIS SERVER INFORMATION ==========")
print("Redis Version:", server_info["redis_version"])
print("Operating System:", server_info["os"])
print("Redis Mode:", server_info["redis_mode"])

print("\n========== MEMORY INFORMATION ==========")
print("Used Memory:", memory_info["used_memory_human"])
print("Peak Memory:", memory_info["used_memory_peak_human"])


========== REDIS SERVER INFORMATION ==========
Redis Version: 7.0.15
Operating System: Linux 6.6.159+ x86_64
Redis Mode: standalone

========== MEMORY INFORMATION ==========
Used Memory: 1.02M
Peak Memory: 1.02M


In [ ]:
print("===== REDIS KEY TYPE VERIFICATION =====")

for product_id in ["P001", "P002", "P003", "P004"]:
    key = f"product:{product_id}"

    print("Key:", key)
    print("Redis Data Type:", r.type(key))
    print("----------------------------")

print("Total Product Records:",
      len(list(r.scan_iter(match="product:*"))))

===== REDIS KEY TYPE VERIFICATION =====
Key: product:P001
Redis Data Type: hash
----------------------------
Key: product:P002
Redis Data Type: hash
----------------------------
Key: product:P003
Redis Data Type: hash
----------------------------
Key: product:P004
Redis Data Type: hash
----------------------------
Total Product Records: 4


In [ ]:

# STEP 3.1: CREATE SHOPPING CARTS

# Create shopping cart for User U1001
r.hset("cart:U1001", mapping={
    "P001": 2,
    "P003": 3
})

# Create shopping cart for User U1002
r.hset("cart:U1002", mapping={
    "P002": 1,
    "P004": 2
})

print("Shopping carts created successfully!")

print("\nUser U1001 Cart:")
print(r.hgetall("cart:U1001"))

print("\nUser U1002 Cart:")
print(r.hgetall("cart:U1002"))


Shopping carts created successfully!

User U1001 Cart:
{'P001': '2', 'P003': '3'}

User U1002 Cart:
{'P002': '1', 'P004': '2'}


In [ ]:

# STEP 3.2: RETRIEVE SHOPPING CART INFORMATION

def display_cart(user_id):

    cart_key = f"cart:{user_id}"

    cart_items = r.hgetall(cart_key)

    print(f"\n===== SHOPPING CART: {user_id} =====")

    if not cart_items:
        print("Shopping cart is empty.")
        return

    for product_id, quantity in cart_items.items():

        product_data = r.hgetall(f"product:{product_id}")

        if product_data:
            print("Product:", product_data["name"])
            print("Quantity:", quantity)
            print("Price: RM", product_data["price"])
            print("----------------------------")


# Retrieve both customers' shopping carts

display_cart("U1001")
display_cart("U1002")



===== SHOPPING CART: U1001 =====
Product: Wireless Mouse
Quantity: 2
Price: RM 29.90
----------------------------
Product: USB-C Cable
Quantity: 3
Price: RM 12.00
----------------------------

===== SHOPPING CART: U1002 =====
Product: Mechanical Keyboard
Quantity: 1
Price: RM 89.50
----------------------------
Product: Laptop Stand
Quantity: 2
Price: RM 45.00
----------------------------


In [ ]:

# STEP 3.3: UPDATE PRODUCT QUANTITIES

print("===== BEFORE UPDATE =====")
print("U1001 Cart:", r.hgetall("cart:U1001"))

# Update Wireless Mouse quantity to 4
r.hset("cart:U1001", "P001", 4)

# Update USB-C Cable quantity to 2
r.hset("cart:U1001", "P003", 2)

print("\n===== AFTER UPDATE =====")
print("U1001 Cart:", r.hgetall("cart:U1001"))


===== BEFORE UPDATE =====
U1001 Cart: {'P001': '2', 'P003': '3'}

===== AFTER UPDATE =====
U1001 Cart: {'P001': '4', 'P003': '2'}


In [ ]:

# STEP 3.4: REMOVE A PRODUCT FROM SHOPPING CART

print("===== BEFORE DELETE =====")
print("U1001 Cart:", r.hgetall("cart:U1001"))

# Remove USB-C Cable from User U1001's cart
r.hdel("cart:U1001", "P003")

print("\n===== AFTER DELETE =====")
print("U1001 Cart:", r.hgetall("cart:U1001"))

print("\nProduct P003 removed successfully!")


===== BEFORE DELETE =====
U1001 Cart: {'P001': '4', 'P003': '2'}

===== AFTER DELETE =====
U1001 Cart: {'P001': '4'}

Product P003 removed successfully!


In [ ]:
# STEP 3.5: DELETE AN ENTIRE SHOPPING CART

print("===== BEFORE DELETING CART =====")
print("U1002 Cart:", r.hgetall("cart:U1002"))

# Delete the entire shopping cart
r.delete("cart:U1002")

print("\n===== AFTER DELETING CART =====")
print("U1002 Cart:", r.hgetall("cart:U1002"))

# Verify whether the cart still exists
if r.exists("cart:U1002"):
    print("Shopping cart still exists.")
else:
    print("Shopping cart deleted successfully!")


===== BEFORE DELETING CART =====
U1002 Cart: {}

===== AFTER DELETING CART =====
U1002 Cart: {}
Shopping cart deleted successfully!


In [ ]:

# STEP 4.1: CALCULATE SHOPPING CART TOTAL

def calculate_cart_total(user_id):

    cart_key = f"cart:{user_id}"
    cart_items = r.hgetall(cart_key)

    total = 0

    print(f"===== CART TOTAL FOR {user_id} =====")

    for product_id, quantity in cart_items.items():

        product_key = f"product:{product_id}"
        product_data = r.hgetall(product_key)

        if product_data:
            product_name = product_data["name"]
            price = float(product_data["price"])
            quantity = int(quantity)

            subtotal = price * quantity
            total += subtotal

            print(f"{product_name}")
            print(f"Quantity: {quantity}")
            print(f"Subtotal: RM{subtotal:.2f}")
            print("----------------------------")

    print(f"TOTAL AMOUNT: RM{total:.2f}")

    return total


# Calculate User U1001's shopping cart total
calculate_cart_total("U1001")


===== CART TOTAL FOR U1001 =====
Wireless Mouse
Quantity: 4
Subtotal: RM119.60
----------------------------
TOTAL AMOUNT: RM119.60


119.6

In [ ]:

# Recreate the second customer's shopping cart

r.hset("cart:U1002", mapping={
    "P002": 1,
    "P004": 2
})

print("Shopping cart U1002 recreated successfully!")
print("Cart data:", r.hgetall("cart:U1002"))
# Retrieve all shopping cart keys using SCAN
cart_keys = sorted(r.scan_iter(match="cart:*"))

print("SHOPPING CART KEYS FOUND:")

for key in cart_keys:
    print("Key:", key)
    print("Cart Data:", r.hgetall(key))

print("Total number of shopping carts:", len(cart_keys))

Shopping cart U1002 recreated successfully!
Cart data: {'P002': '1', 'P004': '2'}
SHOPPING CART KEYS FOUND:
Key: cart:U1001
Cart Data: {'P001': '4'}
Key: cart:U1002
Cart Data: {'P002': '1', 'P004': '2'}
Total number of shopping carts: 2


In [ ]:
# STEP 4.2: PRODUCT POPULARITY RANKING

# Create product popularity scores
r.zadd("popular:products", {
    "P001": 15,
    "P002": 25,
    "P003": 10,
    "P004": 20
})

# Display products from most popular to least popular
print("===== PRODUCT POPULARITY RANKING =====")

ranking = r.zrevrange(
    "popular:products",
    0,
    -1,
    withscores=True
)

for rank, (product_id, score) in enumerate(ranking, start=1):

    product_data = r.hgetall(f"product:{product_id}")

    print(f"Rank: {rank}")
    print("Product:", product_data["name"])
    print("Popularity Score:", int(score))
    print("----------------------------")


===== PRODUCT POPULARITY RANKING =====
Rank: 1
Product: Mechanical Keyboard
Popularity Score: 25
----------------------------
Rank: 2
Product: Laptop Stand
Popularity Score: 20
----------------------------
Rank: 3
Product: Wireless Mouse
Popularity Score: 15
----------------------------
Rank: 4
Product: USB-C Cable
Popularity Score: 10
----------------------------


In [ ]:
# STEP 4.3: REDIS TRANSACTION

print("===== BEFORE TRANSACTION =====")

# Check current stock
print("P001 Stock:", r.hget("product:P001", "stock"))

# Start Redis transaction
transaction = r.pipeline(transaction=True)

transaction.multi()

# Reduce Wireless Mouse stock by 2
transaction.hincrby("product:P001", "stock", -2)

# Record the purchase quantity
transaction.hincrby("sales:P001", "quantity_sold", 2)

# Execute all queued commands
results = transaction.execute()

print("\n===== TRANSACTION EXECUTED =====")

print("Updated Stock:", r.hget("product:P001", "stock"))
print("Quantity Sold:", r.hget("sales:P001", "quantity_sold"))

print("\nTransaction completed successfully!")


===== BEFORE TRANSACTION =====
P001 Stock: 150

===== TRANSACTION EXECUTED =====
Updated Stock: 148
Quantity Sold: 2

Transaction completed successfully!


In [ ]:
# STEP 4.4: REDIS TTL - AUTOMATIC CART EXPIRATION

# Create a temporary shopping cart for testing
r.hset("cart:TEST001", mapping={
    "P001": 2,
    "P002": 1
})

# Set the cart expiration time to 60 seconds
r.expire("cart:TEST001", 60)

# Display the shopping cart
print("===== TEMPORARY SHOPPING CART =====")
print("Cart Contents:", r.hgetall("cart:TEST001"))

# Check remaining expiration time
remaining_time = r.ttl("cart:TEST001")

print("\nRemaining TTL:", remaining_time, "seconds")

# Verify whether expiration is enabled
if remaining_time > 0:
    print("Cart expiration is successfully configured!")


===== TEMPORARY SHOPPING CART =====
Cart Contents: {'P001': '2', 'P002': '1'}

Remaining TTL: 60 seconds
Cart expiration is successfully configured!


In [ ]:
# STEP 4.4.2: VERIFY AUTOMATIC CART EXPIRATION

print("===== CART EXPIRATION VERIFICATION =====")

# Check whether the temporary cart still exists
cart_exists = r.exists("cart:TEST001")

print("Cart Exists:", cart_exists)

# Check remaining TTL
print("Remaining TTL:", r.ttl("cart:TEST001"))

if cart_exists == 0:
    print("SUCCESS: The shopping cart expired automatically!")
else:
    print("The shopping cart has not expired yet.")


===== CART EXPIRATION VERIFICATION =====
Cart Exists: 1
Remaining TTL: 13
The shopping cart has not expired yet.


In [ ]:
# STEP 4.5: FINAL REDIS SYSTEM TESTING

print("========== FINAL SYSTEM TEST ==========")

# Test 1: Product records
product_count = len(list(r.scan_iter(match="product:*")))
print("\n1. Total Products:", product_count)

# Test 2: User U1001 shopping cart
print("\n2. User U1001 Cart:")
print(r.hgetall("cart:U1001"))

# Test 3: User U1002 shopping cart
print("\n3. User U1002 Cart Exists:")
print(r.exists("cart:U1002"))

# Test 4: Product popularity ranking
print("\n4. Popular Products:")
print(r.zrevrange("popular:products", 0, -1, withscores=True))

# Test 5: Sales record
print("\n5. Sales Record:")
print(r.hgetall("sales:P001"))

# Test 6: Temporary cart expiration
print("\n6. Expired Test Cart Exists:")
print(r.exists("cart:TEST001"))

print("\n========== SYSTEM TEST COMPLETED ==========")


========== FINAL SYSTEM TEST ==========

1. Total Products: 4

2. User U1001 Cart:
{'P001': '4'}

3. User U1002 Cart Exists:
0

4. Popular Products:
[('P002', 25.0), ('P004', 20.0), ('P001', 15.0), ('P003', 10.0)]

5. Sales Record:
{'quantity_sold': '2'}

6. Expired Test Cart Exists:
0

========== SYSTEM TEST COMPLETED ==========
